# 🕹️ PAC-NEON // Superset + Antigravity Arcade Orchestrator

Este notebook atua como o **Cockpit Interativo** do **Pac-Neon**, unindo:
- ⚡ **Superset CLI (`superset-sh/superset`)**: Motor de Git Worktrees e terminais paralelos para **Claude Code, Gemini CLI, Codex, Cursor e Aider** (`superset snapshot --json`, `superset terminals read`, `superset terminals send`).
- 🧠 **Google Antigravity Bridge (`AGY`)**: Leitura nativa em tempo real dos transcripts em `~/.gemini/antigravity/brain/*/.system_generated/logs/transcript.jsonl` e ConnectRPC.
- 🍒 **Power Pellet Deck, Monitor Anti-Alucinação & Self-Handoff**: Aprove permissões pendentes (`y`/`n`), monitore a saúde de contexto dos agentes e faça Handoff em 1 clique.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pac_neon.superset_bridge import SupersetBridge
from pac_neon.adapters import AdapterRegistry

bridge = SupersetBridge()
registry = AdapterRegistry()

print("⚡ Status do Motor Superset CLI:", bridge.get_host_status())
print("🔌 Cartuchos Registrados no Pac-Neon:", [p["display_name"] for p in registry.list_providers()])

## 🔍 1. Descoberta ao Vivo: Worktrees do Superset + Sessões do Antigravity

In [ ]:
superset_agents = bridge.discover_from_snapshot()
local_agents = registry.discover_all()

print(f"🟢 Agentes em Worktrees do Superset: {len(superset_agents)}")
for ag in superset_agents:
    print(f"  - [{ag.provider.upper()}] {ag.name} | Worktree: {ag.superset_workspace_id} | Repo: {ag.repo_name} | Estado: {ag.game_state}")

print(f"\n🟡 Agentes Locais / Antigravity Brain: {len(local_agents)}")
for ag in local_agents:
    print(f"  - [{ag.provider.upper()}] {ag.name} | Repo: {ag.repo_name} | Estado: {ag.game_state} | Score: {ag.score_xp} PTS")
    print(f"    ▶ Ação: {ag.current_action}")

## 🕹️ 2. Cockpit Visual Pac-Neon (Embutido no Notebook)
Se o servidor `pac-neon start` estiver rodando na porta `8765`, o painel abaixo conecta via WebSocket em tempo real. Caso contrário, ele renderiza o modo interativo autônomo diretamente na célula!

In [ ]:
import urllib.request
from IPython.display import IFrame, HTML, display

SERVER_URL = "http://127.0.0.1:8765"
try:
    urllib.request.urlopen(f"{SERVER_URL}/api/state", timeout=1.0)
    print("🟢 Conectado ao servidor Pac-Neon ao vivo em", SERVER_URL)
    display(IFrame(src=SERVER_URL, width="100%", height=840))
except Exception:
    print("⚡ Servidor na porta 8765 offline — carregando Cockpit Pac-Neon Standalone na célula:")
    html_path = PROJECT_ROOT / "pac_neon" / "static" / "index.html"
    display(IFrame(src=str(html_path), width="100%", height=840))

## ⚡ 3. Comandos Rápidos de Orquestração (Aprovar Permissões e Broadcast)
Use as funções abaixo para aprovar todas as permissões pendentes (`Power Pellets`) ou enviar um comando/contrato para múltiplos agentes de uma só vez.

In [ ]:
import json
import urllib.request

def pac_neon_api(path: str, method: str = "GET", payload: dict = None):
    url = f"http://127.0.0.1:8765{path}"
    data = json.dumps(payload).encode("utf-8") if payload is not None else None
    req = urllib.request.Request(url, data=data, method=method, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=3.0) as resp:
        return json.loads(resp.read().decode("utf-8"))

def approve_all_power_pellets(action: str = "allow"):
    """Aprova todas as permissões pendentes (Superset terminals + PTY + Hooks) sem trocar de janela."""
    state = pac_neon_api("/api/state")
    pending = [a["pending_permission"] for a in state.get("agents", []) if a.get("pending_permission")]
    if not pending:
        print("✅ Nenhuma permissão pendente no momento!")
        return
    for perm in pending:
        res = pac_neon_api(f"/api/permissions/{perm['id']}/resolve", method="POST", payload={"action": action})
        print(f"🟢 Permissão {perm['id']} ({perm['agent_name']} em {perm['repo_name']}) -> {action.upper()}!")

def send_warp_fruit(source_agent_id: str, target_agent_id: str, title: str, content: str, fruit_type: str = "cherry_contract"):
    """Teleporta um Contrato/Spec de um agente (ex: Antigravity) direto para outro (ex: Worktree no Superset)."""
    return pac_neon_api("/api/warp/send", method="POST", payload={
        "source_agent_id": source_agent_id,
        "target_agent_id": target_agent_id,
        "fruit_type": fruit_type,
        "title": title,
        "content": content,
    })

print("🕹️ Funções prontas: approve_all_power_pellets() | send_warp_fruit(...)")